In [0]:
-- Regular view: a saved query, recomputed on every read
CREATE OR REPLACE VIEW adventure_works.gold.sales_kpis_category_quarter_vw AS
SELECT category, quarter, total_revenue, order_count, avg_order_value
FROM adventure_works.gold.sales_kpis_by_category_quarter;

In [0]:
-- Metric view: one governed definition of the KPIs
CREATE OR REPLACE VIEW adventure_works.gold.sales_kpis_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 0.1
source: adventure_works.gold.sales_kpis_by_category_quarter
dimensions:
  - name: category
    expr: category
  - name: quarter
    expr: quarter
measures:
  - name: total_revenue
    expr: SUM(total_revenue)
  - name: order_count
    expr: SUM(order_count)
  - name: avg_order_value
    expr: SUM(total_revenue) / SUM(order_count)
$$;

result


In [0]:
-- Regular view: total_revenue by category, most recent quarter
SELECT category, total_revenue
FROM adventure_works.gold.sales_kpis_category_quarter_vw
WHERE quarter = (SELECT MAX(quarter) FROM adventure_works.gold.sales_kpis_category_quarter_vw)
ORDER BY total_revenue DESC;

category,total_revenue
Mountain,2635891.6162
Touring,2281226.6698
Road,1972657.8021
Standard,244702.0801
Other,43831.6148


In [0]:
-- Metric view: same question — measures must be wrapped in MEASURE()
SELECT category, MEASURE(total_revenue) AS total_revenue
FROM adventure_works.gold.sales_kpis_metrics
WHERE quarter = (SELECT MAX(quarter) FROM adventure_works.gold.sales_kpis_by_category_quarter)
GROUP BY category
ORDER BY total_revenue DESC;

category,total_revenue
Mountain,2635891.6162
Touring,2281226.6698
Road,1972657.8021
Standard,244702.0801
Other,43831.6148
